# 27 · A tool registry: add tools without touching the loop
Level L3 · Part 4 · core · ~30 min · builds on 26

## Goal
Build a `ToolRegistry` that turns typed Python functions into tool schemas and dispatches calls by name.
Register `search_notes`, `calculator` and `write_file` with a decorator, run lesson 26's loop unchanged,
then add a fourth tool with one decorator and no other edits.

## The idea
In lesson 26 each tool lived in three places: the function, a hand-written schema, and two lists
(`TOOLS`, `SCHEMAS`). Rename a parameter in one place and forget another, and the model is told one
thing while the code expects another. A **tool registry** keeps all of it in one place. You write a
normal Python function with type hints and a docstring; the registry reads them to build the schema
(**schema from type hints**) and stores the function under its name in a **dispatch table**, a dict
from name to function. The loop only asks the registry. Think of a phone switchboard: plug in a new
line and calls reach it, without rewiring the board.

## Picture
```mermaid
flowchart LR
    F["@registry.tool<br/>typed function"] --> S[schema from<br/>hints + docstring]
    F --> D[dispatch table<br/>name -> function]
    S --> L[agent loop]
    D --> L
    L -- "tool call: name, args" --> D
```

## Step by step
### 1. Setup from lesson 26
The notes, `search_notes`, the safe `calculator`, the scripted fake model and the loop, unchanged.
This lesson uses only the fake model, so no recorder and no key are needed.

In [1]:
import ast, inspect, json, operator
from collections.abc import Callable
from pathlib import Path
from typing import Annotated, get_args, get_origin, get_type_hints

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def safe_eval(expression: str) -> float:
    """Lesson 26's calculator body: numbers and + - * / only, never eval."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.left), ev(node.right))
        raise ValueError(f"not allowed in a calculation: {ast.dump(node)[:40]}")
    return ev(ast.parse(expression, mode="eval").body)

`ScriptedModel` and `run_agent` from lesson 26, copied without changes. Watch `run_agent`: it takes a
dict of functions and a list of schemas. That is all a registry has to hand it.

In [2]:
def check_history(messages: list[dict]) -> None:
    issued = set()
    for m in messages:
        issued |= {c["id"] for c in m.get("tool_calls", [])}
        if m["role"] == "tool" and m["call_id"] not in issued:
            raise ValueError(f"tool result for unknown call id {m['call_id']!r}")

class ScriptedModel:
    def __init__(self, script: list):
        self.script, self.calls, self.next_id = script, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        check_history(messages)
        self.calls += 1
        if self.calls > 50:
            raise RuntimeError("model called 50 times: the loop never stopped")
        reply = self.script[min(self.calls, len(self.script)) - 1]
        reply = reply(messages) if callable(reply) else json.loads(json.dumps(reply))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        return {"role": "assistant", **reply}

And the loop. Step 3 inside it is the dispatch: `tools[call["name"]]` looks the function up by name.

In [3]:
def run_agent(question: str, model, tools: dict, schemas: list[dict], max_steps: int = 6) -> dict:
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        reply = model(messages, schemas)                      # 1. call the model
        messages.append(reply)                                # its turn goes into the history
        calls = reply.get("tool_calls", [])
        if not calls:                                         # 2. end of turn: it answered
            print(f"step {step}: answer")
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "messages": messages}
        for call in calls:                                    # 3. run every tool it asked for
            result = tools[call["name"]](**call["args"])
            print(f"step {step}: {call['name']}({call['args']}) -> {json.dumps(result)[:60]}")
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"],
                             "content": json.dumps(result)})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "messages": messages}  # 4. limit
print("setup ready:", len(NOTES), "notes")

setup ready: 6 notes


### 2. From a Python type to a JSON Schema type
A tool schema describes arguments in JSON Schema (lesson 20). Python type hints already say the same
thing: `str` is `"string"`, `int` is `"integer"`. A small function maps one to the other, including
lists like `list[str]`. Anything it does not know raises at once, rather than sending the model a
schema that lies.

In [4]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}

def json_type(hint) -> dict:
    if hint in JSON_TYPES:
        return {"type": JSON_TYPES[hint]}
    if get_origin(hint) is list:
        (item,) = get_args(hint)
        return {"type": "array", "items": json_type(item)}
    raise TypeError(f"no JSON Schema type for {hint!r}")

print(json_type(str), json_type(float), json_type(list[str]))

{'type': 'string'} {'type': 'number'} {'type': 'array', 'items': {'type': 'string'}}


### 3. A whole schema from a function
`inspect.signature` lists the parameters and their defaults. `get_type_hints` gives each type. A
parameter with no default is **required**. The docstring becomes the tool's description, and a
parameter's description rides along in `Annotated[str, "..."]`, a type hint with a note attached.
No hint or no docstring is an error: the model would get a tool it cannot use well.

In [5]:
def schema_from_function(fn) -> dict:
    hints = get_type_hints(fn, include_extras=True)           # keep the Annotated notes
    properties, required = {}, []
    for name, param in inspect.signature(fn).parameters.items():
        if name not in hints:
            raise TypeError(f"{fn.__name__}: parameter {name!r} has no type hint")
        hint, note = hints[name], None
        if get_origin(hint) is Annotated:
            hint, note = get_args(hint)[0], get_args(hint)[1]
        properties[name] = json_type(hint) | ({"description": note} if note else {})
        if param.default is inspect.Parameter.empty:
            required.append(name)
    if not inspect.getdoc(fn):
        raise TypeError(f"{fn.__name__}: needs a docstring; it becomes the tool description")
    return {"name": fn.__name__, "description": inspect.getdoc(fn),
            "parameters": {"type": "object", "properties": properties, "required": required}}

Try it on a tiny function first. `limit` has a default, so it is not in `required`.

In [6]:
def demo(word: Annotated[str, "one word to look for"], limit: int = 3) -> list[str]:
    """Find up to `limit` notes containing a word."""

print(json.dumps(schema_from_function(demo), indent=1))

{
 "name": "demo",
 "description": "Find up to `limit` notes containing a word.",
 "parameters": {
  "type": "object",
  "properties": {
   "word": {
    "type": "string",
    "description": "one word to look for"
   },
   "limit": {
    "type": "integer"
   }
  },
  "required": [
   "word"
  ]
 }
}


### 4. The registry
The registry holds the **dispatch table** (`functions`: name to function) and the schemas. Its `tool`
method is a decorator: it builds the schema, refuses a duplicate name, stores both, and returns the
function unchanged so you can still call it directly.

In [7]:
class ToolRegistry:
    def __init__(self):
        self.functions: dict[str, Callable] = {}              # the dispatch table
        self.schemas: list[dict] = []

    def tool(self, fn):
        schema = schema_from_function(fn)
        if schema["name"] in self.functions:
            raise ValueError(f"tool {schema['name']!r} is already registered")
        self.functions[schema["name"]] = fn
        self.schemas.append(schema)
        return fn                                             # still a normal function

    def call(self, name: str, args: dict):
        return self.functions[name](**args)

registry = ToolRegistry()
print("empty registry:", registry.functions, registry.schemas)

empty registry: {} []


### 5. Register Scout's three tools
The docstrings are now the model-facing descriptions, so they say what the tool returns and when to
use it, like lesson 26's hand-written ones did. `write_file` saves into Scout's folder and keeps only
the file name; lesson 28 handles bad paths properly.

In [8]:
SCOUT_DIR = Path("../data/scout_files")
SCOUT_DIR.mkdir(parents=True, exist_ok=True)
for old in SCOUT_DIR.glob("*.txt"):                       # start each run with an empty folder
    old.unlink()

@registry.tool
def search_notes(query: Annotated[str, "one word to look for, e.g. 'Mars'"]) -> list[dict]:
    """Search the user's notes. Returns every note whose text contains the query word
    (case-insensitive), as a list of {id, text}. Use it before answering any question about the notes."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

@registry.tool
def calculator(expression: Annotated[str, "e.g. '(3+4)*2'"]) -> float:
    """Evaluate an arithmetic expression with numbers, + - * / and brackets, e.g. '24*60+39'.
    Use it for every calculation instead of doing arithmetic yourself."""
    return safe_eval(expression)

The third tool takes two required arguments, so its schema has two `required` entries.

In [9]:
@registry.tool
def write_file(filename: Annotated[str, "a file name only, e.g. 'mars.txt'"],
               text: Annotated[str, "the full text to save"]) -> str:
    """Save text to a file in Scout's folder, replacing the file if it exists.
    Returns a short confirmation with the number of characters written."""
    path = SCOUT_DIR / Path(filename).name                    # name only; lesson 28 does real checks
    path.write_text(text)
    return f"saved {len(text)} characters to {path.name}"

print("registered:", list(registry.functions))
print(json.dumps(registry.schemas[2]["parameters"], indent=1))

registered: ['search_notes', 'calculator', 'write_file']
{
 "type": "object",
 "properties": {
  "filename": {
   "type": "string",
   "description": "a file name only, e.g. 'mars.txt'"
  },
  "text": {
   "type": "string",
   "description": "the full text to save"
  }
 },
 "required": [
  "filename",
  "text"
 ]
}


Compare the generated calculator schema with lesson 26's hand-written `parameters`. Same shape, but
now it cannot drift from the function, because it is read from the function.

In [10]:
HAND_WRITTEN_26 = {"type": "object", "required": ["expression"],
                   "properties": {"expression": {"type": "string", "description": "e.g. '(3+4)*2'"}}}
generated = registry.schemas[1]["parameters"]
print("same as lesson 26:", generated == HAND_WRITTEN_26)
print("dispatch by name:", registry.call("calculator", {"expression": "24*60+39"}))

same as lesson 26: True
dispatch by name: 1479


### 6. Lesson 26's loop, fed by the registry
Scout now answers and saves the answer. The fake model searches, calculates, writes a file built from
the calculator's result, then answers. `run_agent` is the exact function from lesson 26: we pass it
`registry.functions` and `registry.schemas`.

In [11]:
QUESTION = "How many minutes long is a day on Mars? Save the answer to mars_day.txt."
def last_result(messages): return json.loads(messages[-1]["content"])

SAVE_SCRIPT = [
    {"tool_calls": [{"name": "search_notes", "args": {"query": "day"}}]},
    {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
    lambda m: {"tool_calls": [{"name": "write_file", "args": {
        "filename": "mars_day.txt", "text": f"A day on Mars lasts about {last_result(m)} minutes."}}]},
    {"text": "A day on Mars lasts about 1479 minutes. I saved that to mars_day.txt."},
]
run = run_agent(QUESTION, ScriptedModel(SAVE_SCRIPT), registry.functions, registry.schemas)
print(run["stop"], "after", run["steps"], "steps |", (SCOUT_DIR / "mars_day.txt").read_text())

step 1: search_notes({'query': 'day'}) -> [{"id": 3, "text": "A day on Mars lasts about 24 hours and 3
step 2: calculator({'expression': '24*60+39'}) -> 1479
step 3: write_file({'filename': 'mars_day.txt', 'text': 'A day on Mars lasts about 1479 minutes.'}) -> "saved 39 characters to mars_day.txt"
step 4: answer
end_of_turn after 4 steps | A day on Mars lasts about 1479 minutes.


### 7. A fourth tool, one decorator, no loop changes
Scout should be able to list what it has saved. We write the function, decorate it, and that is all.
A tool with no parameters gets an empty `properties` and an empty `required` list.

In [12]:
@registry.tool
def list_files() -> list[str]:
    """List the names of the files saved in Scout's folder, sorted A to Z."""
    return sorted(p.name for p in SCOUT_DIR.iterdir() if p.is_file())

print("tools now:", [s["name"] for s in registry.schemas])
print("list_files schema:", registry.schemas[-1]["parameters"])

tools now: ['search_notes', 'calculator', 'write_file', 'list_files']
list_files schema: {'type': 'object', 'properties': {}, 'required': []}


Now ask Scout about its files. Same `run_agent`, same call, one more tool in the registry.

In [13]:
LIST_SCRIPT = [
    {"tool_calls": [{"name": "list_files", "args": {}}]},
    lambda m: {"text": f"Scout has saved: {', '.join(last_result(m))}."},
]
run = run_agent("What files have you saved?", ScriptedModel(LIST_SCRIPT),
                registry.functions, registry.schemas)
print(run["answer"])

step 1: list_files({}) -> ["mars_day.txt"]
step 2: answer
Scout has saved: mars_day.txt.


### 8. The schemas fit Gemini as they are
Lesson 26's `GeminiModel` built each declaration with
`FunctionDeclaration(name=, description=, parameters_json_schema=)`. Building them from the registry
needs no network call, so we can check every schema is accepted offline. Swap these into
`GeminiModel` and the real model sees all four tools.

In [14]:
from google.genai import types

decls = [types.FunctionDeclaration(name=s["name"], description=s["description"],
                                   parameters_json_schema=s["parameters"]) for s in registry.schemas]
for d in decls:
    print(f"{d.name:13} params: {list(d.parameters_json_schema['properties'])}")

search_notes  params: ['query']
calculator    params: ['expression']
write_file    params: ['filename', 'text']
list_files    params: []


google-genai can build a declaration from a function too: `FunctionDeclaration.from_callable_with_api_option`.
It also runs offline. Compare it with ours on `search_notes`.

In [15]:
sdk = types.FunctionDeclaration.from_callable_with_api_option(callable=search_notes, api_option="VERTEX_AI")
print("required:", sdk.parameters.required, "| type:", sdk.parameters.properties["query"].type)
print("param description from Annotated:", sdk.parameters.properties["query"].description)
print("ours:", registry.schemas[0]["parameters"]["properties"]["query"])

required: ['query'] | type: Type.STRING
param description from Annotated: None
ours: {'type': 'string', 'description': "one word to look for, e.g. 'Mars'"}


Same idea, same required list. The SDK's version did not pick up the `Annotated` note, so the model
would get no hint about what a good query looks like. Writing the 20-line version yourself shows
exactly what the model will see, and frameworks (lesson 44) do the same job behind a decorator.

## At work
- **Frameworks and SDKs generate tool schemas from typed functions**, like `from_callable` above and
  the decorators in agent frameworks. Teams still print and review the generated schema, because that
  is the only part the model reads. → lesson 44
- **Every registered tool costs tokens on every call.** Tool definitions are billed as input (Anthropic
  adds a 286-token tool-use system prompt on Sonnet 5.5 and Opus 5.5, plus your schemas). A registry
  makes adding tools easy, so teams keep the set small and the descriptions sharp. → lessons 29 and 54
- **A schema does not guarantee good arguments.** Production loops validate model-supplied arguments
  (often with Pydantic) and turn an unknown tool name or bad arguments into an error result, not a
  crash. → lesson 28
- **The registry is where tests hook in:** each tool is tested on its own, and a snapshot test fails
  when a generated schema changes unexpectedly. → lesson 46

## What just happened
- `json_type` maps Python hints (`str`, `int`, `float`, `bool`, `list[...]`) to JSON Schema types.
- `schema_from_function` reads the signature, hints, defaults, `Annotated` notes and the docstring to
  build a full tool schema. Parameters without a default are required.
- `ToolRegistry.tool` is a decorator that stores the schema and adds the function to a dispatch table.
- Lesson 26's `run_agent` ran unchanged on `registry.functions` and `registry.schemas`: search,
  calculate, write the file, answer.
- Adding `list_files` took one decorator. The loop and the other tools were not touched.

## Common mistakes
**1. A hand-written schema drifts from the function.** Someone renames the calculator's parameter to
`expr` during a refactor. Lesson 26's schema still says `expression`, so the model sends `expression`,
and the call fails in the middle of a run.

In [16]:
def calculator_v2(expr: str) -> float:                         # parameter renamed
    """Evaluate arithmetic."""
    return safe_eval(expr)

HAND_TOOLS = {"calculator": calculator_v2}                     # schema is still HAND_WRITTEN_26
call = {"name": "calculator", "args": {"expression": "24*60+39"}}  # what the model sends, per the schema
HAND_TOOLS[call["name"]](**call["args"])

TypeError: calculator_v2() got an unexpected keyword argument 'expression'

Fix: generate the schema from the function. The schema now says `expr`, so the model is told the
real name.

In [17]:
print(schema_from_function(calculator_v2)["parameters"]["required"])

['expr']


**2. Leaving out a type hint.** Without it there is no type to put in the schema. The registry refuses
the tool when it is registered, which is the right time: at start-up, not mid-conversation.

In [18]:
@registry.tool
def count_notes(word):
    """Count the notes that contain a word."""
    return len(search_notes(word))

TypeError: count_notes: parameter 'word' has no type hint

Fix: add the hints (and a note for the model). The registration now works.

In [19]:
@registry.tool
def count_notes(word: Annotated[str, "one word, e.g. 'Mars'"]) -> int:
    """Count the notes that contain a word (case-insensitive). Returns a number."""
    return len(search_notes(word))

print(registry.call("count_notes", {"word": "Mars"}), "| tools:", list(registry.functions))

3 | tools: ['search_notes', 'calculator', 'write_file', 'list_files', 'count_notes']


## Try it
Make a new `ToolRegistry` called `my_tools`. Register a `search_notes` that takes
`query: Annotated[str, "..."]` and `max_results: int = 2` and returns at most that many notes. Check
that `max_results` is in `properties` but not in `required`. Then run `run_agent` with a fake model that
calls `search_notes` with only `{"query": "Mars"}` and answers with the number of notes it got back.

In [20]:
# your code here

## Key terms
- **tool registry** — one object that holds every tool's function and schema; tools are added with a decorator and the loop reads from it, so adding a tool needs no loop change.
- **dispatch table** — a dict from tool name to function, used to run the tool the model named.
- **schema from type hints** — building a tool's JSON Schema from a Python function's parameters, type hints, defaults and docstring instead of writing it by hand.
- **`Annotated`** — a type hint with extra data attached, e.g. `Annotated[str, "one word"]`; here the extra data is the parameter's description for the model.